# 05 X-Window Preprocessing for 2026-06 Final Test

Prepare cleaned data and split artifacts for a new Linear Regression X-window experiment.

This notebook follows the cleaning decisions from `02_preprocessing.ipynb`, but writes to a separate output folder so the existing Week 4 baseline artifacts are not overwritten.

Final test month is fixed to `2026-06`.


## 0. Setup


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "x_window_202606"
SPLIT_DIR = PROCESSED_DATA_DIR / "splits"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV = PROCESSED_DATA_DIR / "crmls_xwindow_cleaned.csv"
SPLIT_PLAN_CSV = PROCESSED_DATA_DIR / "x_window_split_plan.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DATA_DIR}")
print(f"Cleaned CSV: {CLEANED_CSV}")
print(f"Split plan CSV: {SPLIT_PLAN_CSV}")


Project root: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction
Raw data dir: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/raw
Cleaned CSV: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/x_window_202606/crmls_xwindow_cleaned.csv
Split plan CSV: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/x_window_202606/x_window_split_plan.csv


## 1. Load Raw Data

Each file is loaded with a `source_file` column so records remain traceable after combining monthly data.


In [2]:
def load_crmls_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["source_file"] = path.name
    return df

csv_files = sorted(RAW_DATA_DIR.glob("CRMLSSold*.csv"))

raw_df = pd.concat([load_crmls_file(path) for path in csv_files], ignore_index=True)

print(f"Loaded files: {len(csv_files)}")
print([path.name for path in csv_files])
print(f"Raw shape: {raw_df.shape}")

Loaded files: 30
['CRMLSSold202401_filled.csv', 'CRMLSSold202402_filled.csv', 'CRMLSSold202403_filled.csv', 'CRMLSSold202404_filled.csv', 'CRMLSSold202405_filled.csv', 'CRMLSSold202406_filled.csv', 'CRMLSSold202407_filled.csv', 'CRMLSSold202408.csv', 'CRMLSSold202409.csv', 'CRMLSSold202410.csv', 'CRMLSSold202411.csv', 'CRMLSSold202412.csv', 'CRMLSSold202501_filled.csv', 'CRMLSSold202502.csv', 'CRMLSSold202503.csv', 'CRMLSSold202504.csv', 'CRMLSSold202505.csv', 'CRMLSSold202506.csv', 'CRMLSSold202507.csv', 'CRMLSSold202508.csv', 'CRMLSSold202509.csv', 'CRMLSSold202510.csv', 'CRMLSSold202511.csv', 'CRMLSSold202512.csv', 'CRMLSSold202601.csv', 'CRMLSSold202602.csv', 'CRMLSSold202603.csv', 'CRMLSSold202604.csv', 'CRMLSSold202605.csv', 'CRMLSSold202606.csv']
Raw shape: (660950, 83)


## 2. Filter Project Scope and Target

- The project scope:  `Residential` + `SingleFamilyResidence`.
- The target is `ClosePrice`, so rows without a valid positive close price are dropped.


In [3]:
df = raw_df.copy()

# Parse target and date fields.
df["ClosePrice"] = pd.to_numeric(df["ClosePrice"], errors="coerce")

transaction_date_cols = [
    "CloseDate",
    "ListingContractDate",
    "PurchaseContractDate",
    "ContractStatusChangeDate",
]
for col in transaction_date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "DaysOnMarket" in df.columns:
    df["DaysOnMarket"] = pd.to_numeric(df["DaysOnMarket"], errors="coerce")

df["close_month"] = df["CloseDate"].dt.to_period("M")

# Target/date validity.
df = df[df["CloseDate"].notna()]
df = df[df["ClosePrice"].notna()]
df = df[df["ClosePrice"] > 0]

# Task-specific property filter.
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"].astype(str).str.replace(" ", "", regex=False) == "SingleFamilyResidence")
].copy()

df.shape

(333361, 84)

## 3. Deduplicate Listings

By EDA, we found duplicated `ListingKey` values.

In [4]:
rows_before = len(df)

df = df.sort_values(["ListingKey", "CloseDate", "source_file"])
df = df.drop_duplicates(subset=["ListingKey"], keep="last")

print(f"Duplicate ListingKey rows removed: {rows_before - len(df):,}")
print(df.shape)


Duplicate ListingKey rows removed: 285
(333076, 84)


## 4. Define Feature Decisions

In [5]:
target = "ClosePrice"
id_col = "ListingKey"
metadata_cols = [id_col, "CloseDate", "close_month", "source_file"]

continuous_numeric_cols = [
    "LivingArea",
    "LotSizeSquareFeet",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "AssociationFee"
]

# Count / ordinal numeric predictors.
# These are numeric, but they are discrete counts rather than continuous measurements.
count_numeric_cols = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "GarageSpaces",
    "ParkingTotal",
    "Stories"
]

# Combined numeric list used for model features and missing flags.
numeric_cols = continuous_numeric_cols + count_numeric_cols

boolean_cols = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN",
]

categorical_cols = [
    "City",
    "CountyOrParish",
    "PostalCode",  # standardized to 5-digit ZIP during cleaning
    "MLSAreaMajor",
    "Levels",
    "Flooring",
    "HighSchoolDistrict",
]

# Extra fields used for preprocessing quality checks. These are kept for auditability,
# but they are not part of numeric_cols/categorical_cols unless explicitly listed above.
quality_check_cols = [
    "LotSizeAcres",
    "ListingContractDate",
    "PurchaseContractDate",
    "ContractStatusChangeDate",
    "DaysOnMarket",
]

# Keep only columns that exist in the current raw files.
metadata_cols = [col for col in metadata_cols if col in df.columns]
numeric_cols = [col for col in numeric_cols if col in df.columns]
boolean_cols = [col for col in boolean_cols if col in df.columns]
categorical_cols = [col for col in categorical_cols if col in df.columns]
quality_check_cols = [col for col in quality_check_cols if col in df.columns]

selected_cols = list(dict.fromkeys(metadata_cols + [target] + numeric_cols + categorical_cols + boolean_cols + quality_check_cols))
clean_df = df[selected_cols].copy()

print(f"Clean base shape: {clean_df.shape}")
print(f"Quality check columns: {quality_check_cols}")

Clean base shape: (333076, 33)
Quality check columns: ['LotSizeAcres', 'ListingContractDate', 'PurchaseContractDate', 'ContractStatusChangeDate', 'DaysOnMarket']


## 5. Clean Numeric, Boolean, Location, ZIP, Lot Size, and Date Quality Values

This section applies row-level preprocessing decisions based on the added EDA checks:

- convert numeric columns to numeric dtype
- remove clearly non-California records
- flag invalid coordinates and set bad coordinates to missing
- compare `LotSizeAcres` with `LotSizeSquareFeet` using 43,560 square feet per acre
- standardize `PostalCode` to the first 5-digit ZIP for modeling
- run date-order sanity checks and keep issue flags
- convert impossible numeric values to missing
- convert boolean fields to 0/1 before creating missing flags
- add missing-value flags after boolean normalization
- clean categorical text values

In [6]:
clean_df = clean_df.copy()

# 1. Numeric cleaning: convert invalid strings to NaN.
for col in numeric_cols + ["LotSizeAcres", "DaysOnMarket"]:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# 2. Drop records that are explicitly outside the California modeling scope.
# These are not just bad coordinates; the location labels themselves are non-CA.
if "CountyOrParish" in clean_df.columns:
    non_ca_location = clean_df["CountyOrParish"].isin(["Foreign Country", "Other State"])
    print(f"Dropping non-California location rows: {non_ca_location.sum():,}")
    clean_df = clean_df.loc[~non_ca_location].copy()

# 3. Coordinate quality flag.
ca_lat_min, ca_lat_max = 32.0, 42.5
ca_lon_min, ca_lon_max = -125.0, -113.0

if {"Latitude", "Longitude"}.issubset(clean_df.columns):
    coords_present = clean_df["Latitude"].notna() & clean_df["Longitude"].notna()
    invalid_coords = coords_present & (
        ~clean_df["Latitude"].between(ca_lat_min, ca_lat_max)
        | ~clean_df["Longitude"].between(ca_lon_min, ca_lon_max)
    )

    clean_df["invalid_coordinates_flag"] = invalid_coords.astype(int)
    clean_df.loc[invalid_coords, ["Latitude", "Longitude"]] = np.nan
    print(f"Rows with coordinates present: {int(coords_present.sum()):,}")
    print(f"Invalid coordinate rows flagged: {int(invalid_coords.sum()):,}")
else:
    clean_df["invalid_coordinates_flag"] = 0

# 4. Convert impossible numeric values to missing.
if "LivingArea" in clean_df.columns:
    clean_df.loc[clean_df["LivingArea"] <= 0, "LivingArea"] = np.nan
if "BedroomsTotal" in clean_df.columns:
    clean_df.loc[clean_df["BedroomsTotal"] < 0, "BedroomsTotal"] = np.nan
if "BathroomsTotalInteger" in clean_df.columns:
    clean_df.loc[clean_df["BathroomsTotalInteger"] < 0, "BathroomsTotalInteger"] = np.nan
if "LotSizeSquareFeet" in clean_df.columns:
    clean_df.loc[clean_df["LotSizeSquareFeet"] <= 0, "LotSizeSquareFeet"] = np.nan
if "LotSizeAcres" in clean_df.columns:
    clean_df.loc[clean_df["LotSizeAcres"] <= 0, "LotSizeAcres"] = np.nan
if "YearBuilt" in clean_df.columns:
    current_year = pd.Timestamp.today().year
    clean_df.loc[~clean_df["YearBuilt"].between(1800, current_year + 1), "YearBuilt"] = np.nan
for col in ["GarageSpaces", "ParkingTotal", "Stories", "AssociationFee"]:
    if col in clean_df.columns:
        clean_df.loc[clean_df[col] < 0, col] = np.nan

# 5. Lot size unit consistency check.
# If acres and square feet disagree materially, flag the row and set LotSizeSquareFeet to missing.
# If square feet is missing but acres is valid, derive square feet from acres and flag that derivation.
clean_df["lot_size_unit_mismatch_flag"] = 0
clean_df["lot_size_sqft_derived_from_acres_flag"] = 0

if {"LotSizeAcres", "LotSizeSquareFeet"}.issubset(clean_df.columns):
    acres = clean_df["LotSizeAcres"]
    sqft = clean_df["LotSizeSquareFeet"]
    expected_sqft = acres * 43560

    both_available = acres.notna() & sqft.notna()
    abs_sqft_diff = (sqft - expected_sqft).abs()
    abs_pct_diff = abs_sqft_diff / expected_sqft
    lot_size_mismatch = both_available & (abs_sqft_diff > 50) & (abs_pct_diff > 0.02)

    clean_df.loc[lot_size_mismatch, "lot_size_unit_mismatch_flag"] = 1
    clean_df.loc[lot_size_mismatch, "LotSizeSquareFeet"] = np.nan

    missing_sqft_with_acres = clean_df["LotSizeSquareFeet"].isna() & clean_df["LotSizeAcres"].notna() & ~lot_size_mismatch
    clean_df.loc[missing_sqft_with_acres, "LotSizeSquareFeet"] = clean_df.loc[missing_sqft_with_acres, "LotSizeAcres"] * 43560
    clean_df.loc[missing_sqft_with_acres, "lot_size_sqft_derived_from_acres_flag"] = 1

    print(f"Lot size unit mismatch rows flagged: {int(lot_size_mismatch.sum()):,}")
    print(f"LotSizeSquareFeet rows derived from acres: {int(missing_sqft_with_acres.sum()):,}")

# 6. PostalCode standardization: use only the first 5-digit ZIP for modeling.
if "PostalCode" in clean_df.columns:
    postal_raw = clean_df["PostalCode"].astype("string").str.strip()
    postal_missing = postal_raw.isna() | postal_raw.eq("")
    postal5 = postal_raw.str.extract(r"(\d{5})", expand=False)
    postal_format_issue = (~postal_missing) & postal5.isna()

    clean_df["PostalCode_format_issue_flag"] = postal_format_issue.astype(int)
    clean_df["PostalCode"] = postal5.fillna("__missing__")

    print(f"PostalCode rows without extractable 5-digit ZIP: {int(postal_format_issue.sum()):,}")
else:
    clean_df["PostalCode_format_issue_flag"] = 0

# 7. Date order sanity checks. Keep flags for auditability; do not drop rows here.
for col in ["ListingContractDate", "PurchaseContractDate", "ContractStatusChangeDate"]:
    if col in clean_df.columns:
        clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")

date_sanity_rows = []
date_issue_flags = []
date_order_checks = [
    ("listing_after_purchase_flag", "ListingContractDate", "PurchaseContractDate", "ListingContractDate <= PurchaseContractDate"),
    ("purchase_after_close_flag", "PurchaseContractDate", "CloseDate", "PurchaseContractDate <= CloseDate"),
    ("listing_after_close_flag", "ListingContractDate", "CloseDate", "ListingContractDate <= CloseDate"),
]

for flag_col, start_col, end_col, check_name in date_order_checks:
    clean_df[flag_col] = 0
    if start_col in clean_df.columns and end_col in clean_df.columns:
        comparable = clean_df[start_col].notna() & clean_df[end_col].notna()
        violations = comparable & (clean_df[start_col] > clean_df[end_col])
        clean_df.loc[violations, flag_col] = 1
        date_issue_flags.append(flag_col)
        date_sanity_rows.append({
            "check": check_name,
            "n_compared": int(comparable.sum()),
            "n_violations": int(violations.sum()),
            "violation_rate": float(violations.sum() / comparable.sum()) if comparable.sum() else np.nan,
        })

if "DaysOnMarket" in clean_df.columns:
    clean_df["negative_days_on_market_flag"] = clean_df["DaysOnMarket"].lt(0).fillna(False).astype(int)
    date_issue_flags.append("negative_days_on_market_flag")
    comparable = clean_df["DaysOnMarket"].notna()
    violations = comparable & (clean_df["DaysOnMarket"] < 0)
    date_sanity_rows.append({
        "check": "DaysOnMarket >= 0",
        "n_compared": int(comparable.sum()),
        "n_violations": int(violations.sum()),
        "violation_rate": float(violations.sum() / comparable.sum()) if comparable.sum() else np.nan,
    })
else:
    clean_df["negative_days_on_market_flag"] = 0
    date_issue_flags.append("negative_days_on_market_flag")

clean_df["date_order_issue_flag"] = clean_df[date_issue_flags].max(axis=1) if date_issue_flags else 0

date_sanity_summary = pd.DataFrame(date_sanity_rows)
display(date_sanity_summary)

# 8. Boolean cleaning. Normalize first so missing flags capture both original missing
# values and unexpected strings that become NaN during normalization.
def normalize_bool(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"true", "t", "yes", "y", "1"}:
        return 1.0
    if text in {"false", "f", "no", "n", "0"}:
        return 0.0
    return np.nan

for col in boolean_cols:
    clean_df[col] = clean_df[col].map(normalize_bool)

# 9. Missing flags after all row-level corrections and boolean normalization.
missing_flag_cols = []
for col in numeric_cols + boolean_cols:
    flag_col = f"{col}_missing"
    clean_df[flag_col] = clean_df[col].isna().astype(int)
    missing_flag_cols.append(flag_col)

# 10. Categorical text cleaning.
for col in categorical_cols:
    clean_df[col] = (
        clean_df[col]
        .astype("string")
        .str.strip()
        .replace("", "__missing__")
        .fillna("__missing__")
    )

cleaning_summary = pd.DataFrame({
    "column": numeric_cols + boolean_cols + categorical_cols,
    "missing_rate_after_cleaning": clean_df[numeric_cols + boolean_cols + categorical_cols].isna().mean().values,
})

display(cleaning_summary.sort_values("missing_rate_after_cleaning", ascending=False))
print(f"Rows after cleaning: {len(clean_df):,}")

Dropping non-California location rows: 16


Rows with coordinates present: 332,959
Invalid coordinate rows flagged: 75
Lot size unit mismatch rows flagged: 173
LotSizeSquareFeet rows derived from acres: 19


PostalCode rows without extractable 5-digit ZIP: 0


,check,n_compared,n_violations,violation_rate
0,ListingContractDate <= PurchaseContractDate,332922,203,0.000610
1,PurchaseContractDate <= CloseDate,332922,196,0.000589
2,ListingContractDate <= CloseDate,333060,46,0.000138
3,DaysOnMarket >= 0,333060,42,0.000126


,column,missing_rate_after_cleaning
5,AssociationFee,0.302348
10,Stories,0.126130
13,AttachedGarageYN,0.116517
12,PoolPrivateYN,0.095526
11,ViewYN,0.089654
15,NewConstructionYN,0.073083
8,GarageSpaces,0.037273
1,LotSizeSquareFeet,0.018114
9,ParkingTotal,0.001282
0,LivingArea,0.000934


Rows after cleaning: 333,060


## 6. Export Cleaned Data

In [7]:
export_df = clean_df.copy()
export_df["log_ClosePrice"] = np.log1p(export_df[target])
export_df.to_csv(CLEANED_CSV, index=False)

print(f"Saved cleaned data: {CLEANED_CSV}")

Saved cleaned data: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/x_window_202606/crmls_xwindow_cleaned.csv


## 7. X-Window Split Setup

Use `2026-06` as the final test month. Compare candidate training windows on the same historical evaluation cutoffs, then use the selected X once on the final test month.

Candidate windows are intentionally configurable. This draft uses `X = 3, 6, 9, 12, 18` months.


In [8]:
available_months = sorted(clean_df["close_month"].dropna().unique())
final_test_month = pd.Period("2026-06", freq="M")

if final_test_month not in available_months:
    raise ValueError(f"Final test month {final_test_month} is not present in cleaned data.")

candidate_windows = [3, 6, 9, 12, 18]
# Same historical cutoffs for every candidate X, spread across the available period.
historical_eval_months = [
    pd.Period("2025-07", freq="M"),
    pd.Period("2025-09", freq="M"),
    pd.Period("2025-11", freq="M"),
    pd.Period("2026-02", freq="M"),
    pd.Period("2026-05", freq="M"),
]

missing_eval_months = [month for month in historical_eval_months if month not in available_months]
if missing_eval_months:
    raise ValueError(f"Historical evaluation months missing from cleaned data: {missing_eval_months}")

max_x = max(candidate_windows)
first_eval_idx = min(available_months.index(month) for month in historical_eval_months)
if first_eval_idx < max_x:
    raise ValueError(
        f"Earliest historical cutoff does not have enough lookback for X={max_x}. "
        f"Earliest cutoff index: {first_eval_idx}."
    )

print(f"Available months: {available_months[0]} to {available_months[-1]}")
print(f"Final test month: {final_test_month}")
print(f"Candidate X windows: {candidate_windows}")
print(f"Historical eval months: {', '.join(str(month) for month in historical_eval_months)}")


def get_train_months(end_month, x_months):
    end_idx = available_months.index(end_month)
    start_idx = end_idx - x_months
    if start_idx < 0:
        raise ValueError(f"Not enough lookback before {end_month} for X={x_months}.")
    return available_months[start_idx:end_idx]


def make_split(data, x_months, eval_month):
    train_months = get_train_months(eval_month, x_months)
    train_df = data[data["close_month"].isin(train_months)].copy()
    eval_df = data[data["close_month"] == eval_month].copy()
    return train_df, eval_df, train_months


Available months: 2024-01 to 2026-06
Final test month: 2026-06
Candidate X windows: [3, 6, 9, 12, 18]
Historical eval months: 2025-07, 2025-09, 2025-11, 2026-02, 2026-05


## 8. Save Split Plan and Evaluation/Test Sets


In [9]:
split_plan_rows = []

for x in candidate_windows:
    for eval_month in historical_eval_months:
        train_df, eval_df, train_months = make_split(clean_df, x, eval_month)
        split_plan_rows.append({
            "split_type": "historical_eval",
            "X_train_months": x,
            "train_month_start": str(train_months[0]),
            "train_month_end": str(train_months[-1]),
            "train_months": ", ".join(str(month) for month in train_months),
            "eval_month": str(eval_month),
            "train_rows": len(train_df),
            "eval_rows": len(eval_df),
        })

for x in candidate_windows:
    train_df, test_df, train_months = make_split(clean_df, x, final_test_month)
    split_plan_rows.append({
        "split_type": "final_test_candidate",
        "X_train_months": x,
        "train_month_start": str(train_months[0]),
        "train_month_end": str(train_months[-1]),
        "train_months": ", ".join(str(month) for month in train_months),
        "eval_month": str(final_test_month),
        "train_rows": len(train_df),
        "eval_rows": len(test_df),
    })

split_plan = pd.DataFrame(split_plan_rows)
split_plan.to_csv(SPLIT_PLAN_CSV, index=False)

# Clear only this experiment's split folder.
for old_path in SPLIT_DIR.glob("*.csv"):
    old_path.unlink()

for eval_month in historical_eval_months:
    eval_df = clean_df[clean_df["close_month"] == eval_month].copy()
    eval_df.to_csv(SPLIT_DIR / f"eval_{eval_month}_cleaned.csv", index=False)

test_df = clean_df[clean_df["close_month"] == final_test_month].copy()
test_df.to_csv(SPLIT_DIR / "test_2026-06_cleaned.csv", index=False)

print(f"Saved split plan: {SPLIT_PLAN_CSV}")
print(f"Saved evaluation/test files to: {SPLIT_DIR}")
display(split_plan)


Saved split plan: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/x_window_202606/x_window_split_plan.csv
Saved evaluation/test files to: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/x_window_202606/splits


,split_type,X_train_months,train_month_start,train_month_end,train_months,eval_month,train_rows,eval_rows
0,historical_eval,3,2025-04,2025-06,"2025-04, 2025-05, 2025-06",2025-07,35326,12110
1,historical_eval,3,2025-06,2025-08,"2025-06, 2025-07, 2025-08",2025-09,35244,11449
2,historical_eval,3,2025-08,2025-10,"2025-08, 2025-09, 2025-10",2025-11,34911,9720
3,historical_eval,3,2025-11,2026-01,"2025-11, 2025-12, 2026-01",2026-02,27651,8542
4,historical_eval,3,2026-02,2026-04,"2026-02, 2026-03, 2026-04",2026-05,31736,12014
5,historical_eval,6,2025-01,2025-06,"2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2...",2025-07,62906,12110
6,historical_eval,6,2025-03,2025-08,"2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2...",2025-09,69477,11449
7,historical_eval,6,2025-05,2025-10,"2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2...",2025-11,70475,9720
8,historical_eval,6,2025-08,2026-01,"2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2...",2026-02,62562,8542
9,historical_eval,6,2025-11,2026-04,"2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2...",2026-05,59387,12014


## 9. Summary


In [10]:
summary = pd.DataFrame([{
    "raw_rows": len(raw_df),
    "cleaned_rows": len(clean_df),
    "available_month_start": str(available_months[0]),
    "available_month_end": str(available_months[-1]),
    "final_test_month": str(final_test_month),
    "candidate_X_values": ", ".join(map(str, candidate_windows)),
    "historical_eval_months": ", ".join(str(month) for month in historical_eval_months),
    "cleaned_csv": str(CLEANED_CSV.relative_to(PROJECT_ROOT)),
    "split_plan_csv": str(SPLIT_PLAN_CSV.relative_to(PROJECT_ROOT)),
    "split_dir": str(SPLIT_DIR.relative_to(PROJECT_ROOT)),
}]).T.rename(columns={0: "value"})

display(summary)


,value
raw_rows,660950
cleaned_rows,333060
available_month_start,2024-01
available_month_end,2026-06
final_test_month,2026-06
candidate_X_values,"3, 6, 9, 12, 18"
historical_eval_months,"2025-07, 2025-09, 2025-11, 2026-02, 2026-05"
cleaned_csv,data/processed/x_window_202606/crmls_xwindow_c...
split_plan_csv,data/processed/x_window_202606/x_window_split_...
split_dir,data/processed/x_window_202606/splits
